# 05 — Salon source cross-agreement (Stage 3c)

Spec §6.7 names this as a **methods sub-finding**, not a sensitivity analysis: report counts and distance distribution between matched salons in the two sources, and produce union and intersection layers used downstream as sensitivity inputs.

Inputs: outputs of notebooks 03 (Google Places) and 04 (OSM Overpass). Both are restricted to NE LSOAs.

Match rule (DEC-014 family):
    distance ≤ 50 m **and** rapidfuzz token-set name score ≥ 85.

Outputs:
- `data/processed/salons_intersection_<date>.gpkg` — matched pairs only.
- `data/processed/salons_union_<date>.gpkg` — Google ∪ OSM with provenance label.
- `audit_logs/source_agreement_<date>.csv` — per-LSOA Google / OSM / matched counts.

## 0. Pre-flight

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config, salons_dedupe

config.ensure_dirs()
audit.verify_manifest()

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
TODAY

## 1. Load both salon layers

In [ ]:
google_files = sorted(config.DATA_PROCESSED.glob("salons_google_*.gpkg"))
osm_files = sorted(config.DATA_PROCESSED.glob("salons_osm_*.gpkg"))
if not google_files:
    raise RuntimeError("No Google salons GeoPackage found — run notebook 03 first.")
if not osm_files:
    raise RuntimeError("No OSM salons GeoPackage found — run notebook 04 first.")

google_path = google_files[-1]
osm_path = osm_files[-1]

google_gdf = gpd.read_file(google_path)
osm_gdf = gpd.read_file(osm_path)
print(f"Loaded {google_path.name}: {len(google_gdf):,} Google places")
print(f"Loaded {osm_path.name}: {len(osm_gdf):,} OSM features")

## 2. Match by spatial proximity + fuzzy name

In [ ]:
result = salons_dedupe.match_salons(
    google_gdf,
    osm_gdf,
    max_distance_m=salons_dedupe.DEFAULT_MAX_DISTANCE_M,
    name_score_min=salons_dedupe.DEFAULT_NAME_SCORE_MIN,
)
summary = result.summary
for k, v in summary.items():
    print(f"  {k:>22s}: {v}")

In [ ]:
result.matched.head(10)

## 3. Distance and name-score distributions on matched pairs

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(result.matched["match_distance_m"], bins=20)
axes[0].set_xlabel("Match distance (m)")
axes[0].set_ylabel("Pairs")
axes[0].set_title("Spatial offset between matched Google ↔ OSM salons")
axes[1].hist(result.matched["name_score"], bins=20)
axes[1].set_xlabel("Name score (rapidfuzz token_set_ratio)")
axes[1].set_ylabel("Pairs")
axes[1].set_title("Name-string agreement on matched pairs")
plt.tight_layout()
plt.show()

## 4. Per-LA breakdown

In [ ]:
google_per_la = google_gdf.groupby("lad_code").size().rename("n_google")
osm_per_la = osm_gdf.groupby("lad_code").size().rename("n_osm")
matched_google_per_la = (
    google_gdf.loc[google_gdf["place_id"].isin(result.matched["place_id"])]
    .groupby("lad_code")
    .size()
    .rename("n_matched")
)

by_la = (
    pd.DataFrame({"lad_code": list(config.LA_CODES_NE)})
    .merge(google_per_la, on="lad_code", how="left")
    .merge(osm_per_la, on="lad_code", how="left")
    .merge(matched_google_per_la, on="lad_code", how="left")
)
by_la[["n_google", "n_osm", "n_matched"]] = by_la[["n_google", "n_osm", "n_matched"]].fillna(0).astype(int)
by_la["lad_name"] = by_la["lad_code"].map(config.LA_NAMES_NE)
by_la["jaccard"] = by_la["n_matched"] / (by_la["n_google"] + by_la["n_osm"] - by_la["n_matched"]).replace(0, pd.NA)
by_la = by_la[["lad_code", "lad_name", "n_google", "n_osm", "n_matched", "jaccard"]].set_index("lad_code")
by_la.loc["TOTAL"] = by_la.sum(numeric_only=True)
by_la.loc["TOTAL", "lad_name"] = "NE total"
by_la.loc["TOTAL", "jaccard"] = (
    by_la.loc["TOTAL", "n_matched"]
    / (by_la.loc["TOTAL", "n_google"] + by_la.loc["TOTAL", "n_osm"] - by_la.loc["TOTAL", "n_matched"])
)
by_la

## 5. Per-LSOA agreement (audit log)

In [ ]:
lsoa = gpd.read_file(sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))[-1])
lsoa_agreement = salons_dedupe.lsoa_level_agreement(google_gdf, osm_gdf, lsoa)
agreement_path = config.AUDIT_LOGS / f"source_agreement_{TODAY}.csv"
lsoa_agreement.to_csv(agreement_path, index=False)
print(f"Wrote per-LSOA agreement table: {agreement_path.relative_to(config.REPO_ROOT)}")
lsoa_agreement.describe()

## 6. Build union and intersection layers

In [ ]:
# Reduce both inputs to a small common schema before unioning so the
# union table is self-describing.
g = google_gdf[["place_id", "name", "address", "lad_code", "imd_quintile", "geometry"]].copy()
g["osm_id"] = pd.NA
o = osm_gdf[["osm_id", "name", "addr_full", "lad_code", "imd_quintile", "geometry"]].rename(
    columns={"addr_full": "address"}
).copy()
o["place_id"] = pd.NA

intersection_gdf, union_gdf = salons_dedupe.build_union_intersection(
    g, o, result.matched
)
print(f"Intersection rows: {len(intersection_gdf):,}")
print(f"Union rows       : {len(union_gdf):,}")
print("Source breakdown in union:")
print(union_gdf["source"].value_counts().to_string())

In [ ]:
out_int = config.DATA_PROCESSED / f"salons_intersection_{TODAY}.gpkg"
out_union = config.DATA_PROCESSED / f"salons_union_{TODAY}.gpkg"
intersection_gdf.to_file(out_int, driver="GPKG", layer="salons_intersection")
union_gdf.to_file(out_union, driver="GPKG", layer="salons_union")

for out_path, inputs, notes in (
    (out_int, (google_path, osm_path), "Salons present in BOTH Google Places and OSM (matched pairs)."),
    (out_union, (google_path, osm_path), "Union of Google Places and OSM salon enumerations, with source label."),
):
    audit.write_provenance_sidecar(
        audit.Provenance(
            output_path=out_path,
            inputs=inputs,
            notes=notes,
            random_seed=config.RANDOM_SEED,
        )
    )
    print("Wrote:", out_path.name)

## 7. Headline statement for the manuscript methods sub-finding

A single line summarising agreement at NE level — the kind of sentence that goes into the manuscript.

In [ ]:
n_g, n_o, n_m = summary["n_google"], summary["n_osm"], summary["n_matched"]
jaccard = n_m / (n_g + n_o - n_m) if (n_g + n_o - n_m) else float("nan")
print(
    f"Across the 12 NE LAs, Google Places identified {n_g} commercial tanning premises and "
    f"OSM identified {n_o}. Spatial-fuzzy matching (≤50 m + name-score ≥85) paired "
    f"{n_m} of these. Median offset between matched pairs was "
    f"{summary['median_match_distance_m']:.1f} m. Jaccard = {jaccard:.2f}."
)

## Done

Outputs ready for Stage 6 sensitivity analyses: `salons_union_<date>.gpkg` (default for the analysis), `salons_intersection_<date>.gpkg` (sensitivity #1 in the spec list), and the per-LSOA agreement table in `audit_logs/`.